# Week 1 — BillSum preparation and model audit

This notebook records my BillSum data checks and Qwen model setup audit.

The checks cover:
- Dataset selection: 3,000 training, 500 tuning, and 500 test examples.
- Shared IDs and text fingerprints across the three groups.
- Tokenization and truncation of bills and reference summaries.
- Trainable parameter counts for full fine-tuning, LoRA, and DoRA.
- The existing BillSum tests: 14 passed.

The model audit ran on the CPU. T4 availability was checked separately.
No model training or final test-set scoring was performed.

The first environment checks were run before switching to the GPU session.
Later outputs record the GPU-enabled environment.

The evidence archive preserves the failed audit, the torchao compatibility
fix, and the successful retry.

In [2]:
import sys
from pathlib import Path

print("Python version:", sys.version.split()[0])
print("Current folder:", Path.cwd())
print("Notebook purpose: Jainish - Week 1 - BillSum")

Python version: 3.13.18
Current folder: /kaggle/working
Notebook purpose: Jainish - Week 1 - BillSum


In [3]:
from importlib.metadata import version, PackageNotFoundError

packages = ["torch", "transformers", "peft", "bitsandbytes", "pyarrow"]

for package in packages:
    try:
        print(package, ":", version(package))
    except PackageNotFoundError:
        print(package, ": not installed")

torch : 2.11.0+cpu
transformers : 5.20.2
peft : 0.21.2
bitsandbytes : not installed
pyarrow : 23.0.1


In [1]:
!nvidia-smi

Wed Nov 11 19:51:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

# Let our model code see only the first GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("PyTorch version:", torch.__version__)
print("GPU usable by PyTorch:", torch.cuda.is_available())
print("GPUs visible to PyTorch:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU usable by PyTorch: True
GPUs visible to PyTorch: 1
GPU name: Tesla T4


In [3]:
import subprocess

subprocess.run(
    [
        "git", "clone",
        "https://github.com/devjp99/lora-on-a-budget.git",
        "/kaggle/working/lora-on-a-budget"
    ],
    check=True
)

Cloning into '/kaggle/working/lora-on-a-budget'...


CompletedProcess(args=['git', 'clone', 'https://github.com/devjp99/lora-on-a-budget.git', '/kaggle/working/lora-on-a-budget'], returncode=0)

In [5]:
from pathlib import Path

project = Path("/kaggle/working/lora-on-a-budget")

print("Project folder exists:", project.is_dir())

for item in sorted(project.iterdir()):
    if not item.name.startswith("."):
        print(item.name)

Project folder exists: True
README.md
configs
docs
manifests
notebooks
pyproject.toml
requirements-billsum.txt
requirements.txt
results
scripts
src
tests


In [6]:
print("BILLSUM SCRIPTS:")
for file in sorted((project / "scripts").glob("*billsum*")):
    print(file.name)

print("\nREQUESTED SOFTWARE:")
print((project / "requirements-billsum.txt").read_text())

BILLSUM SCRIPTS:
audit_billsum.py
make_billsum_manifest.py
pilot_billsum.py

REQUESTED SOFTWARE:
# Isolated decoder candidate environment; CPU-tested, T4 compatibility still pending.
# Keep separate from Kunj's requirements.txt until both owners validate one environment.
torch==2.5.1
transformers==4.46.3
peft==0.14.0
pyarrow==18.1.0
bitsandbytes==0.45.0
accelerate==1.15.0
huggingface-hub==0.36.2
pandas==2.2.3
PyYAML==6.0.3
pytest==9.2.0



In [7]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Current Python:", sys.version.split()[0])

for package in ["torch", "transformers", "peft", "pyarrow"]:
    try:
        print(package, ":", version(package))
    except PackageNotFoundError:
        print(package, ": not installed")

print("\nPROJECT INSTALLATION SETTINGS:")
print((project / "pyproject.toml").read_text())

Current Python: 3.13.16
torch : 2.11.0+cu130
transformers : 5.20.2
peft : 0.21.2
pyarrow : 23.0.1

PROJECT INSTALLATION SETTINGS:
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "lora_budget"
version = "0.1.0"
description = "LoRA on a Budget: controlled fine-tuning study on a single 16 GB GPU (CMPE 252)"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]



In [8]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--editable", str(project),
        "--no-deps"
    ],
    check=True
)

Obtaining file:///kaggle/working/lora-on-a-budget
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lora_budget (pyproject.toml): started
  Building editable for lora_budget (pyproject.toml): finished with status 'done'
  Created wheel for lora_budget: filename=lora_budget-0.1.0-0.editable-py3-none-any.whl size=1306 sha256=91825598442f39e01eb4d86dcffff847309f9938f303f801fd140db6ed5562ed
  Stored in directory: /tmp/pip-ephem-wheel-cache-dw5ohx9h/wheels/fc/7f/c1/d386dabe01896a581402da5602269710baff4d17b8d9b94a97
Successfully b

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--editable', '/kaggle/working/lora-on-a-budget', '--no-deps'], returncode=0)

In [9]:
subprocess.run(
    [
        sys.executable,
        "scripts/make_billsum_manifest.py",
        "--verify-existing"
    ],
    cwd=str(project),
    check=True
)

Locked raw files and all manifests verified


CompletedProcess(args=['/usr/bin/python3', 'scripts/make_billsum_manifest.py', '--verify-existing'], returncode=0)

In [10]:
print("Manifest files:")

for file in sorted((project / "manifests").rglob("*")):
    if file.is_file():
        print(file.relative_to(project))

Manifest files:
manifests/banking77/label_map.json
manifests/banking77/meta.json
manifests/banking77/test.csv
manifests/banking77/train.csv
manifests/banking77/tune.csv
manifests/billsum/meta.json
manifests/billsum/test.csv
manifests/billsum/train.csv
manifests/billsum/tune.csv


In [11]:
import pandas as pd

train_manifest = pd.read_csv(
    project / "manifests" / "billsum" / "train.csv"
)

print("Number of training records:", len(train_manifest))
print("Column names:", train_manifest.columns.tolist())

display(train_manifest.head(3))

Number of training records: 3000
Column names: ['id', 'row', 'text_sha256']


,id,row,text_sha256
0,train-00011,11,32935922cb2897629959f42ca822e690141d1627e5072d...
1,train-00016,16,201f9568100d521a71143b2c9dd81791d17aa8ac42c6e7...
2,train-00020,20,a2fd50671d09ec9fa33c4626912ba94c0e832bd4d250ca...


In [12]:
tune_manifest = pd.read_csv(
    project / "manifests/billsum/tune.csv"
)
test_manifest = pd.read_csv(
    project / "manifests/billsum/test.csv"
)

print("Training records:", len(train_manifest))
print("Tuning records:", len(tune_manifest))
print("Test records:", len(test_manifest))

train_ids = set(train_manifest["id"])
tune_ids = set(tune_manifest["id"])
test_ids = set(test_manifest["id"])

print("Train/tune shared IDs:", len(train_ids & tune_ids))
print("Train/test shared IDs:", len(train_ids & test_ids))
print("Tune/test shared IDs:", len(tune_ids & test_ids))


Training records: 3000
Tuning records: 500
Test records: 500
Train/tune shared IDs: 0
Train/test shared IDs: 0
Tune/test shared IDs: 0


In [13]:
train_hashes = set(train_manifest["text_sha256"])
tune_hashes = set(tune_manifest["text_sha256"])
test_hashes = set(test_manifest["text_sha256"])

print("Train/tune shared text fingerprints:",
      len(train_hashes & tune_hashes))

print("Train/test shared text fingerprints:",
      len(train_hashes & test_hashes))

print("Tune/test shared text fingerprints:",
      len(tune_hashes & test_hashes))

Train/tune shared text fingerprints: 0
Train/test shared text fingerprints: 0
Tune/test shared text fingerprints: 0


In [14]:
import json

metadata = json.loads(
    (project / "manifests/billsum/meta.json").read_text()
)

print(json.dumps(metadata, indent=2))

{
  "dataset": "FiscalNote/billsum",
  "revision": "3d8510441c06a3d9dfb32eb0d7f80151730bcc4f",
  "license": "CC0-1.0",
  "subset_seed": 2026,
  "selection_algorithm": "python.Random shuffle train then test",
  "raw_sha256": {
    "train": "7ca4e515ad7502ef579a055c2d64e6dfa0d64200017c15bca00f323a34cd2ae3",
    "test": "2733cb656a2a6fbff0fc012ca50bbf159615ff6451c0adb229884bbd474780b3"
  },
  "raw_counts": {
    "train": 18949,
    "test": 3269
  },
  "schemas": {
    "train": {
      "text": "object",
      "summary": "object",
      "title": "object"
    },
    "test": {
      "text": "object",
      "summary": "object",
      "title": "object"
    }
  },
  "planned_counts": {
    "train": 3000,
    "tune": 500,
    "test": 500
  },
  "final_counts": {
    "train": 3000,
    "tune": 500,
    "test": 500
  },
  "duplicates_removed": {
    "train": 0,
    "tune": 0
  },
  "repeats_within_split": {
    "train": 1,
    "tune": 0,
    "test": 0
  },
  "manifest_sha256": {
    "train": "e5506

In [15]:
print(
    (project / "scripts/make_billsum_manifest.py").read_text()
)

"""Download the documented U.S. Parquet release and freeze subset membership."""
import argparse
import csv
import json
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from lora_budget.data.billsum import select_rows
from lora_budget.data.hashing import sha256_file
import pandas as pd
import shutil


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--revision', default='main')
    ap.add_argument('--verify-existing', action='store_true', help='Restore raw files using the locked revision and verify existing manifests')
    ap.add_argument('--raw-dir', default='data/raw/billsum')
    ap.add_argument('--out-dir', default='manifests/billsum')
    args = ap.parse_args()
    out, raw = Path(args.out_dir), Path(args.raw_dir)
    if not args.verify_existing and out.exists() and any(out.iterdir()):
        raise FileExistsError('Manifests already exist; verify them, do not silently regenerate')
    locked = json.loads((out/'meta.json').read_text())

In [16]:
raw_train = pd.read_parquet(
    project / "data/raw/billsum/train.parquet"
)

source_row = int(train_manifest.iloc[0]["row"])
example = raw_train.iloc[source_row]

print("Selected example:", train_manifest.iloc[0]["id"])
print("Source row:", source_row)

print("\nTITLE:")
print(example["title"])

print("\nBILL TEXT — first 1,500 characters:")
print(example["text"][:1500])

print("\nREFERENCE SUMMARY:")
print(example["summary"])

Selected example: train-00011
Source row: 11

TITLE:
To amend the Internal Revenue Code of 1986 to allow businesses to expense qualified security devices.

BILL TEXT — first 1,500 characters:
SECTION 1. SHORT TITLE.

    This Act may be cited as the ``Public Safety and Protection 
Investment Act of 2003''.

SEC. 2. BUSINESS DEDUCTION FOR PURCHASE AND INSTALLATION OF SECURITY 
              DEVICES.

    (a) In General.--Part VI of subchapter B of chapter 1 of the 
Internal Revenue Code of 1986 (relating to itemized deductions for 
individuals and corporations) is amended by inserting after section 
179A the following new section:

``SEC. 179B. SECURITY DEVICE PURCHASES.

    ``(a) Allowance of Deduction.--A taxpayer may elect to treat the 
cost of any qualifying security device as an expense which is not 
chargeable to capital account. Any cost so treated shall be allowed as 
a deduction for the taxable year in which such device is placed in 
service.
    ``(b) Definitions.--For purpos

In [17]:
print(
    (project / "scripts/audit_billsum.py").read_text()
)

"""Verify real model tensors and preprocessing without training or test scoring."""
import gc
import importlib.metadata
import platform
import json
from pathlib import Path
import torch
import yaml
from transformers import AutoTokenizer
from lora_budget.data.billsum import load_split,encode_example
from lora_budget.models.qwen import build_model,audit_parameters


def main():
    cfg=yaml.safe_load(Path('configs/billsum/full_ft.yaml').read_text())
    revision=cfg['model']['revision']
    tok=AutoTokenizer.from_pretrained(cfg['model']['name'],revision=revision)
    report=dict(model_revision=revision,device='cpu',is_gpu_pilot=False,parameters={},preprocessing={},
                environment=dict(python=platform.python_version(),packages={n:importlib.metadata.version(n) for n in ['torch','transformers','peft','pandas','pyarrow']}))
    for split in ('train','tune'):
        encoded=[encode_example(r,tok) for r in load_split(split)]
        report['preprocessing'][split]=dict(count=len(e

In [ ]:
# Remove the optional package that caused our LoRA audit to fail.
# This keeps torch (PyTorch) installed.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True
)

In [25]:
from datetime import datetime, timezone
import shutil

run_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
evidence = Path("/kaggle/working/week1_evidence") / run_name
evidence.mkdir(parents=True, exist_ok=True)

report_path = project / "results/billsum/cpu_audit.json"

if report_path.exists():
    shutil.copy2(report_path, evidence / "previous_cpu_audit.json")

with (evidence / "audit_log.txt").open("w") as log:
    result = subprocess.run(
        [sys.executable, "scripts/audit_billsum.py"],
        cwd=str(project),
        stdout=log,
        stderr=subprocess.STDOUT
    )

print((evidence / "audit_log.txt").read_text())
print("Exit code:", result.returncode)
print("Evidence folder:", evidence)

if result.returncode == 0:
    shutil.copy2(report_path, evidence / "new_cpu_audit.json")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 490.39it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 633.55it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 655.94it/s]
{
  "full_ft": {
    "trainable": 494032768,
    "parameter_dtypes": [
      "torch.float32"
    ]
  },
  "lora": {
    "trainable": 4399104,
    "parameter_dtypes": [
      "torch.float16",
      "torch.float32"
    ]
  },
  "dora": {
    "trainable": 4703232,
    "parameter_dtypes": [
      "torch.float16",
      "torch.float32"
    ]
  }
}

Exit code: 0
Evidence folder: /kaggle/working/week1_evidence/20261111_201331


In [29]:
report = json.loads(
    (evidence / "new_cpu_audit.json").read_text()
)

for split in ("train", "tune"):
    stats = report["preprocessing"][split]

    print(f"\n{split.upper()}")
    print("Examples checked:", stats["count"])
    print("Bill inputs shortened:", stats["source_truncated"])
    print("Target summaries shortened:", stats["target_truncated"])


TRAIN
Examples checked: 3000
Bill inputs shortened: 2953
Target summaries shortened: 1099

TUNE
Examples checked: 500
Bill inputs shortened: 488
Target summaries shortened: 164


In [31]:
import sys
import inspect

source_folder = str(project / "src")

if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

from lora_budget.data.billsum import encode_example

print(inspect.getsource(encode_example))

def encode_example(row, tokenizer, max_prompt=1024, max_target=256):
    if tokenizer.eos_token_id is None or max_target < 1:
        raise ValueError('An EOS token and positive target budget are required')
    enc = lambda text: tokenizer.encode(text, add_special_tokens=False)
    prefix, suffix = enc(PREFIX), enc(SUFFIX)
    budget = max_prompt - len(prefix) - len(suffix)
    if budget < 1:
        raise ValueError('Prompt budget must leave room for bill text')
    source, target = enc(row['text']), enc(row['summary'])
    prompt = prefix + source[:budget] + suffix
    answer = target[:max_target-1] + [tokenizer.eos_token_id]
    return dict(input_ids=prompt+answer, attention_mask=[1]*(len(prompt)+len(answer)),
                labels=[-100]*len(prompt)+answer, prompt_ids=prompt,
                reference=row['summary'], source_tokens=len(source), target_tokens=len(target)+1,
                source_truncated=len(source)>budget, target_truncated=len(target)+1>max_target)



In [32]:
from lora_budget.data.billsum import PREFIX, SUFFIX

print("BEFORE THE BILL:")
print(repr(PREFIX))

print("\nAFTER THE BILL:")
print(repr(SUFFIX))

BEFORE THE BILL:
'Summarize the following bill:\n\n'

AFTER THE BILL:
'\n\nSummary:\n'


In [33]:
with (evidence / "billsum_tests_log.txt").open("w") as log:
    test_result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests/test_billsum.py"],
        cwd=str(project),
        stdout=log,
        stderr=subprocess.STDOUT
    )

print((evidence / "billsum_tests_log.txt").read_text())
print("Test exit code:", test_result.returncode)

..............                                                           [100%]
14 passed in 10.42s

Test exit code: 0


In [38]:
import importlib.metadata

environment = {
    "python": sys.version,
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=str(project),
        text=True
    ).strip(),
    "packages": {
        name: importlib.metadata.version(name)
        for name in [
            "torch", "transformers", "peft",
            "pandas", "pyarrow", "pytest"
        ]
    },
    "audit_device": "cpu",
    "note": "Removed incompatible torchao 0.10.0 before successful audit."
}

(evidence / "environment.json").write_text(
    json.dumps(environment, indent=2)
)

archive = shutil.make_archive(
    "/kaggle/working/jainish_week1_evidence",
    "zip",
    root_dir="/kaggle/working",
    base_dir="week1_evidence"
)

print("Evidence ZIP created:", archive)

Evidence ZIP created: /kaggle/working/jainish_week1_evidence.zip
